# NB04b — MGnify PGLS validation (Exploratory)

**Status:** Exploratory analysis.

**Goal:** Aggregate MGnify MAG density to genus level and run PGLS, mirroring NB04 for SPIRE.

**Steps:**
1. Aggregate `ko_per_mb_primary` to genus by median.
2. Run PGLS if phylogenetic tree is available.
3. Report genus-level associations.

**Output:** `data/mgnify_pgls_validation.csv`.


In [1]:
print("NB04b executing — PGLS validation on MGnify (exploratory).")

NB04b executing — PGLS validation on MGnify (exploratory).


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

# pgls_utils from comprehensive_metal_ecology
_REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(_REPO_ROOT / 'comprehensive_metal_ecology' / 'scripts'))
try:
    from pgls_utils import run_pgls, pgls_results_table, fdr_correct
    pgls_available = True
except ImportError as e:
    print(f"Warning: pgls_utils not available ({e}) — PGLS analysis will be skipped.")
    pgls_available = False

DATA_DIR = Path.cwd().parent / 'data'
CME_DATA = _REPO_ROOT / 'comprehensive_metal_ecology' / 'data'

In [3]:
# MGnify feature matrix
df = pd.read_csv(DATA_DIR / 'mgnify_mag_feature_matrix.csv')

print(f"MGnify feature matrix: {len(df):,} MAGs")
print(f"Columns: {list(df.columns)}")

# Check if genus column exists
if 'genus' not in df.columns:
    print("Note: No 'genus' column in MGnify data — PGLS will not be run.")
    print("This is expected if MGnify taxonomy was not added during feature matrix construction.")
else:
    print(f"Genera in MGnify data: {df['genus'].nunique()}")

MGnify feature matrix: 8,849 MAGs
Columns: ['genome_id', 'sample_accession', 'lineage', 'biome_name', 'biome_lineage', 'length', 'gc_content', 'n_metal_types', 'total_metal_genes', 'domain', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'latitude', 'longitude', 'completeness', 'contamination', 'n_ko_primary', 'n_ko_cofactor', 'n_ko_metabolism', 'n_ko_resistance', 'n_ko_sensing', 'n_ko_transport', 'n_ko_unknown', 'ko_per_mb_primary', 'ko_per_mb_resistance', 'ko_per_mb_transport', 'ko_per_mb_sensing', 'ko_per_mb_metabolism', 'ko_per_mb_cofactor', 'PF1_As', 'PF1_Cd', 'PF1_Cr', 'PF1_Cu', 'PF1_Hg', 'PF1_Pb']
Genera in MGnify data: 3518


In [4]:
# Aggregate to genus level (if genus column exists)
if 'genus' not in df.columns:
    print("Skipping genus aggregation: 'genus' column not in data.")
    genus_df = None
else:
    genus_df = df.groupby('genus').agg(
        ko_per_mb_primary=('ko_per_mb_primary', 'median'),
        PF1_Cu=('PF1_Cu', 'median'),
        n_mags=('genome_id', 'count'),
    ).reset_index()
    
    print(f"Genera with ≥1 MAG: {len(genus_df)}")
    print(genus_df[['genus', 'ko_per_mb_primary', 'PF1_Cu', 'n_mags']].head())

Genera with ≥1 MAG: 3518
             genus  ko_per_mb_primary    PF1_Cu  n_mags
0  0-14-0-80-60-11          24.108656  0.083262       2
1   01-FULL-49-22b          22.338016       NaN       1
2  1-14-0-10-31-34           8.893606       NaN       1
3  1-14-2-50-31-20          19.534158       NaN       2
4   12-FULL-67-14b          13.451867  0.063020       3


In [5]:
# PGLS analysis (if genus data exists and pgls_utils is available)
if genus_df is not None and pgls_available:
    TREE_PATH = CME_DATA / 'gtdb_bac_genus_pruned.tree'
    if not TREE_PATH.exists():
        print(f"Warning: Phylogenetic tree not found at {TREE_PATH} — PGLS will be skipped.")
    else:
        # Z-score features
        genus_df['ko_per_mb_primary_z'] = stats.zscore(genus_df['ko_per_mb_primary'], nan_policy='omit')
        genus_df['PF1_Cu_z'] = stats.zscore(genus_df['PF1_Cu'], nan_policy='omit')
        
        pgls_out = run_pgls(
            df=genus_df,
            tree_path=TREE_PATH,
            response='PF1_Cu_z',
            predictors=['ko_per_mb_primary_z'],
            taxon_col='genus',
        )
        
        results_df = pgls_results_table([pgls_out])
        if 'p_value' in results_df.columns:
            results_df['p_value_fdr'] = fdr_correct(results_df['p_value'].dropna())
        
        results_df.to_csv(DATA_DIR / 'mgnify_pgls_validation.csv', index=False)
        print("PGLS results:")
        print(results_df.to_string(index=False))
else:
    if genus_df is None:
        print("Genus aggregation not available.")
    elif not pgls_available:
        print("pgls_utils not available.")
    print("PGLS analysis skipped.")

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


PGLS results:
label response           predictor   n  lambda_est      beta       SE    t_stat  p_value       r2  delta_aic_vs_null  p_value_fdr
      PF1_Cu_z ko_per_mb_primary_z 444      0.9795 -0.047093 0.041052 -1.147162 0.251935 0.002877               0.68     0.251935
